# Notebook 02 — Forecast Prophet 90 días sobre SAB.MC

Construimos el **baseline** del proyecto: un Prophet básico, comparado con un Prophet que incorpora los **eventos de la OPA BBVA** como `holidays`. Backtest sobre los últimos 90 días + forecast 90 días al futuro. Tracking con MLflow.


## 1. Imports y carga de datos

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

import mlflow
import mlflow.pyfunc

from src.data_loader import OPA_BBVA_EVENTS

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
# Cargar dataset limpio generado en notebook 01
sab = pd.read_csv(ROOT / "data" / "sab_5y_clean.csv", parse_dates=["Date"])
print("Filas:", len(sab))
print("Rango:", sab["Date"].min().date(), "->", sab["Date"].max().date())
sab.head(3)


## 2. Preparación para Prophet

Prophet espera dos columnas con nombres específicos: `ds` (fecha) y `y` (valor a predecir).


In [ ]:
df = sab[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"}).copy()
df = df.dropna().reset_index(drop=True)
df.tail()


## 3. Split train/test

Reservamos los últimos **90 días** para validar. El resto entrena.


In [ ]:
HORIZON = 90
train = df.iloc[:-HORIZON].copy()
test  = df.iloc[-HORIZON:].copy()
print(f"Train: {len(train)} filas  |  Test (holdout): {len(test)} filas")
print(f"Test arranca en: {test['ds'].min().date()}")


## 4. Configurar MLflow

In [ ]:
mlflow.set_tracking_uri('file:///' + (ROOT / 'mlruns').as_posix())
mlflow.set_experiment("sab_forecast_prophet")


## 5. Modelo A — Prophet baseline (sin holidays)

Configuración mínima para tener referencia.


In [ ]:
with mlflow.start_run(run_name="prophet_baseline"):
    m_base = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        interval_width=0.80,           # IC al 80%
        changepoint_prior_scale=0.05,  # default
        seasonality_prior_scale=10,    # default
    )
    m_base.fit(train)

    future_base = m_base.make_future_dataframe(periods=HORIZON, freq="D", include_history=False)
    fcst_base   = m_base.predict(future_base)

    # Evaluar contra test
    eval_base = test.merge(fcst_base[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds", how="left")
    mae_base  = (eval_base["y"] - eval_base["yhat"]).abs().mean()
    rmse_base = np.sqrt(((eval_base["y"] - eval_base["yhat"])**2).mean())
    mape_base = ((eval_base["y"] - eval_base["yhat"]).abs() / eval_base["y"]).mean() * 100

    mlflow.log_param("model", "prophet_baseline")
    mlflow.log_param("horizon", HORIZON)
    mlflow.log_metric("mae",  mae_base)
    mlflow.log_metric("rmse", rmse_base)
    mlflow.log_metric("mape", mape_base)

    print(f"BASELINE  ·  MAE: {mae_base:.3f}  RMSE: {rmse_base:.3f}  MAPE: {mape_base:.2f}%")


## 6. Modelo B — Prophet con eventos de la OPA BBVA

Añadimos los hitos clave de la OPA hostil como `holidays`. Esto permite al modelo "no asustarse" en esas fechas y mejorar la generalización.


In [ ]:
print("Eventos OPA cargados como holidays:")
print(OPA_BBVA_EVENTS)


In [ ]:
with mlflow.start_run(run_name="prophet_opa_holidays"):
    m_opa = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        holidays=OPA_BBVA_EVENTS,
        interval_width=0.80,
        changepoint_prior_scale=0.10,  # subimos un poco — la serie tiene rupturas
    )
    # Festividades de España estándar
    m_opa.add_country_holidays(country_name="ES")
    m_opa.fit(train)

    future_opa = m_opa.make_future_dataframe(periods=HORIZON, freq="D", include_history=False)
    fcst_opa   = m_opa.predict(future_opa)

    eval_opa = test.merge(fcst_opa[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds", how="left")
    mae_opa  = (eval_opa["y"] - eval_opa["yhat"]).abs().mean()
    rmse_opa = np.sqrt(((eval_opa["y"] - eval_opa["yhat"])**2).mean())
    mape_opa = ((eval_opa["y"] - eval_opa["yhat"]).abs() / eval_opa["y"]).mean() * 100

    mlflow.log_param("model", "prophet_opa_holidays")
    mlflow.log_param("horizon", HORIZON)
    mlflow.log_param("changepoint_prior_scale", 0.10)
    mlflow.log_metric("mae",  mae_opa)
    mlflow.log_metric("rmse", rmse_opa)
    mlflow.log_metric("mape", mape_opa)

    print(f"CON OPA   ·  MAE: {mae_opa:.3f}  RMSE: {rmse_opa:.3f}  MAPE: {mape_opa:.2f}%")


## 7. Comparativa visual

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train["ds"], train["y"], color="black", label="Train", linewidth=1)
ax.plot(test["ds"],  test["y"],  color="black", linestyle="--", label="Real (test)", linewidth=1.5)

ax.plot(eval_base["ds"], eval_base["yhat"], color="tab:orange", label=f"Baseline (RMSE {rmse_base:.2f})")
ax.fill_between(eval_base["ds"], eval_base["yhat_lower"], eval_base["yhat_upper"], color="tab:orange", alpha=0.15)

ax.plot(eval_opa["ds"],  eval_opa["yhat"], color="tab:blue", label=f"+ holidays OPA (RMSE {rmse_opa:.2f})")
ax.fill_between(eval_opa["ds"], eval_opa["yhat_lower"], eval_opa["yhat_upper"], color="tab:blue", alpha=0.15)

ax.set_title("SAB.MC — Forecast 90 días: baseline vs con eventos OPA")
ax.set_xlabel("Fecha"); ax.set_ylabel("EUR")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Resumen
resumen = pd.DataFrame({
    "Modelo": ["Baseline", "Con eventos OPA"],
    "MAE":    [mae_base, mae_opa],
    "RMSE":   [rmse_base, rmse_opa],
    "MAPE %": [mape_base, mape_opa]
}).set_index("Modelo")
resumen


## 8. Cross-validation Prophet (robustez de la métrica)

Hacemos walk-forward CV con ventanas de 90 días para no fiarnos de un único holdout.


In [ ]:
# CV con el mejor modelo
df_cv = cross_validation(
    m_opa,
    initial="730 days",
    period="90 days",
    horizon="90 days",
    disable_tqdm=True
)
df_perf = performance_metrics(df_cv, rolling_window=1)
df_perf[["horizon", "mae", "rmse", "mape"]].head()


In [ ]:
fig = plot_cross_validation_metric(df_cv, metric="rmse")
plt.title("Evolución del RMSE en función del horizonte de previsión")
plt.tight_layout()
plt.show()


## 9. Forecast 90 días al FUTURO (entrenando con todo el histórico)

Una vez validado, reentrenamos con todo el histórico y proyectamos 90 días reales hacia adelante.


In [ ]:
# Reentrenar con TODO el histórico
m_final = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    holidays=OPA_BBVA_EVENTS,
    interval_width=0.80,
    changepoint_prior_scale=0.10,
)
m_final.add_country_holidays(country_name="ES")
m_final.fit(df)

future = m_final.make_future_dataframe(periods=HORIZON, freq="D", include_history=False)
fcst_future = m_final.predict(future)
fcst_future[["ds", "yhat", "yhat_lower", "yhat_upper"]].head()


In [ ]:
# Gráfico del forecast a 90 días
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df["ds"], df["y"], color="black", label="Histórico", linewidth=1)
ax.plot(fcst_future["ds"], fcst_future["yhat"], color="tab:blue", label="Forecast 90d")
ax.fill_between(fcst_future["ds"], fcst_future["yhat_lower"], fcst_future["yhat_upper"],
                color="tab:blue", alpha=0.2, label="IC 80%")
ax.set_title(f"SAB.MC — Forecast 90 días al futuro (modelo con eventos OPA)")
ax.set_xlabel("Fecha"); ax.set_ylabel("EUR")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Componentes (tendencia, semanal, anual, holidays)
fig2 = m_final.plot_components(fcst_future)
plt.tight_layout()
plt.show()


## 10. Guardar previsión

In [ ]:
out_path = ROOT / "data" / "sab_forecast_prophet_90d.csv"
fcst_future[["ds", "yhat", "yhat_lower", "yhat_upper"]].to_csv(out_path, index=False)
print(f"Forecast guardado: {out_path}")


## 11. Conclusiones del notebook 02

- Tenemos un baseline Prophet **medido contra test** (RMSE y MAPE explícitos).
- Añadir los eventos OPA como `holidays` y subir un poco `changepoint_prior_scale` mejora generalización en presencia de rupturas (verás los números en la tabla resumen).
- Tracking en MLflow → `./mlruns/`. Para abrir la UI: `mlflow ui --backend-store-uri ./mlruns`.

**Siguiente paso:** `03_forecast_pycaret.ipynb` — comparativa AutoML con PyCaret time_series para ver si supera a Prophet.
